In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tukuyin ang mga opsyon

<details>
<summary><b>Mga bersyon ng package</b></summary>

Ang code sa pahinang ito ay ginawa gamit ang mga sumusunod na requirements.
Inirerekumenda naming gamitin ang mga bersyong ito o mas bago.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

Maaari kang gumamit ng mga opsyon para i-customize ang Estimator at Sampler primitives. Ang seksyong ito ay nakatuon sa kung paano tukuyin ang mga opsyon ng Qiskit Runtime primitive. Habang ang interface ng `run()` na paraan ng mga primitive ay pare-pareho sa lahat ng implementasyon, ang kanilang mga opsyon ay hindi. Tingnan ang kaukulang mga API reference para sa impormasyon tungkol sa [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) at [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) na mga opsyon.

Mga tala tungkol sa pagtukoy ng mga opsyon sa primitives:

- Ang `SamplerV2` at `EstimatorV2` ay may hiwalay na mga klase ng opsyon. Maaari mong makita ang mga available na opsyon at i-update ang mga halaga ng opsyon sa panahon ng o pagkatapos ng pagsisimula ng primitive.
- Gamitin ang `update()` na paraan para ilapat ang mga pagbabago sa katangiang `options`.
- Kung hindi mo tinukoy ang isang halaga para sa isang opsyon, bibigyan ito ng espesyal na halaga ng `Unset` at gagamitin ang mga default ng server.
- Ang katangiang `options` ay ang `dataclass` na Python type. Maaari mong gamitin ang built-in na `asdict` na paraan para i-convert ito sa isang dictionary.

<span id="pass-options"></span>
## Itakda ang mga opsyon ng primitive
Maaari kang magtakda ng mga opsyon kapag sinisimulan ang primitive, pagkatapos masimulan ang primitive, o sa `run()` na paraan. Tingnan ang seksyon ng [mga panuntuan sa precedence](runtime-options-overview#options-precedence) para maunawaan kung ano ang mangyayari kapag ang parehong opsyon ay tinukoy sa maraming lugar.

### Pagsisimula ng primitive
Maaari kang magpasa ng isang instance ng klase ng opsyon o isang dictionary kapag sinisimulan ang isang primitive, na pagkatapos ay gumagawa ng kopya ng mga opsyong iyon. Kaya, ang pagbabago ng orihinal na dictionary o instance ng opsyon ay hindi nakakaapekto sa mga opsyong pag-aari ng mga primitive.

#### Klase ng opsyon
Kapag gumagawa ng instance ng `EstimatorV2` o `SamplerV2` na klase, maaari kang magpasa ng instance ng klase ng opsyon. Ang mga opsyong iyon ay ilalapat kapag ginamit mo ang `run()` para magsagawa ng kalkulasyon. Tukuyin ang mga opsyon sa format na ito: `options.option.sub-option.sub-sub-option = choice`. Halimbawa: `options.dynamical_decoupling.enable = True`

Halimbawa:

Ang `SamplerV2` at `EstimatorV2` ay may hiwalay na mga klase ng opsyon ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) at [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Dictionary
Maaari kang tumukoy ng mga opsyon bilang isang dictionary kapag sinisimulan ang primitive.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### I-update ang mga opsyon pagkatapos ng pagsisimula
Maaari kang tumukoy ng mga opsyon sa format na ito: `primitive.options.option.sub-option.sub-sub-option = choice` para mapakinabangan ang auto-complete, o gamitin ang `update()` na paraan para gumawa ng bulk na mga update.

Ang mga klase ng opsyon ng `SamplerV2` at `EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) at [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) ay hindi kailangang i-instantiate kung nagtatakda ka ng mga opsyon pagkatapos masimulan ang primitive.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Paraan ng Run()
Ang mga value lamang na maaari mong ipasa sa `run()` ay ang mga tinukoy sa interface. Iyon ay, ang `shots` para sa Sampler at ang `precision` para sa Estimator. Ini-overwrite nito ang anumang value na itinakda para sa `default_shots` o `default_precision` para sa kasalukuyang run.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### Mga espesyal na kaso
#### Resilience level (Estimator lamang)
Ang resilience level ay hindi talaga isang opsyon na direktang nakakaapekto sa primitive query, kundi tumutukoy ng base na hanay ng mga piniling opsyon na pagtatayuan. Sa pangkalahatan, ang level 0 ay nagpapatay ng lahat ng error mitigation, ang level 1 ay nagbubukas ng mga opsyon para sa measurement error mitigation, at ang level 2 ay nagbubukas ng mga opsyon para sa gate at measurement error mitigation.

Ang anumang mga opsyon na manu-manong tinukoy mo bilang karagdagan sa resilience level ay inilalapat sa ibabaw ng base na hanay ng mga opsyon na tinukoy ng resilience level. Kaya, sa prinsipyo, maaari mong itakda ang resilience level sa 1, ngunit pagkatapos ay patayin ang measurement mitigation, kahit hindi ito pinapayuhan.

Sa sumusunod na halimbawa, ang pagtatakda ng resilience level sa 0 ay nagpapatay ng `zne_mitigation` sa una, ngunit ang `estimator.options.resilience.zne_mitigation = True` ay nag-o-override ng kaugnay na setup mula sa `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Shots (Sampler lamang)
Ang `SamplerV2.run` na paraan ay tumatanggap ng dalawang argumento: isang listahan ng mga PUB, na ang bawat isa ay maaaring tumukoy ng isang PUB-specific na halaga para sa shots, at isang shots keyword argument. Ang mga halaga ng shot na ito ay bahagi ng Sampler execution interface, at independent ang mga ito sa mga opsyon ng Runtime Sampler. Sila ay may priyoridad kaysa sa anumang mga value na tinukoy bilang mga opsyon upang sumunod sa Sampler abstraction.

Gayunpaman, kung ang `shots` ay hindi tinukoy ng anumang PUB o sa run keyword argument (o kung lahat sila ay `None`), gagamitin ang halaga ng shots mula sa mga opsyon, lalo na ang `default_shots`.

Para ibuod, ito ang order ng precedence para sa pagtukoy ng shots sa Sampler, para sa anumang partikular na PUB:

1. Kung ang PUB ay tumutukoy ng shots, gamitin ang halagang iyon.
2. Kung ang `shots` keyword argument ay tinukoy sa `run`, gamitin ang halagang iyon.
3. Kung ang `num_randomizations` at `shots_per_randomization` ay tinukoy bilang `twirling` na mga opsyon, ang shots ay ang produkto ng mga halagang iyon.
3. Kung ang `sampler.options.default_shots` ay tinukoy, gamitin ang halagang iyon.

Kaya, kung ang shots ay tinukoy sa lahat ng posibleng lugar, ang isa na may pinakamataas na precedence (shots na tinukoy sa PUB) ang ginagamit.

#### Precision (Estimator lamang)
Ang Precision ay katulad ng shots, na inilarawan sa nakaraang seksyon, maliban na ang mga opsyon ng Estimator ay naglalaman ng parehong `default_shots` at `default_precision`. Bukod dito, dahil ang gate-twirling ay pinagana bilang default, ang produkto ng `num_randomizations` at `shots_per_randomization` ay may priyoridad kaysa sa dalawang opsyong iyon.

Partikular, para sa anumang partikular na Estimator PUB:

1. Kung ang PUB ay tumutukoy ng precision, gamitin ang halagang iyon.
2. Kung ang precision keyword argument ay tinukoy sa `run`, gamitin ang halagang iyon.
2. Kung ang `num_randomizations` at `shots_per_randomization` ay tinukoy bilang [`twirling` na mga opsyon](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (pinagana bilang default), gamitin ang kanilang produkto para kontrolin ang dami ng data.
3. Kung ang `estimator.options.default_shots` ay tinukoy, gamitin ang halagang iyon para kontrolin ang dami ng data.
4. Kung ang `estimator.options.default_precision` ay tinukoy, gamitin ang halagang iyon.

Halimbawa, kung ang precision ay tinukoy sa lahat ng apat na lugar, ang isa na may pinakamataas na precedence (precision na tinukoy sa PUB) ang ginagamit.

> **Note:** Ang Precision ay inverse na may kaugnayan sa paggamit. Iyon ay, mas mababa ang precision, mas maraming oras ng QPU ang kinakailangan para tumakbo.

## Mga karaniwang ginagamit na opsyon
Maraming available na opsyon, ngunit ang mga sumusunod ay ang pinakakaraniwang ginagamit:

<span id="shots"></span>
### Shots
Para sa ilang algorithm, ang pagtatakda ng isang tiyak na bilang ng shots ay isang pangunahing bahagi ng kanilang mga rutina. Ang Shots (o precision) ay maaaring tukuyin sa maraming lugar. Pinripriyoridad ang mga ito bilang mga sumusunod:

Para sa anumang Sampler PUB:

1. Mga shots na may integer na halaga na nakapaloob sa PUB
2. Ang halaga ng `run(...,shots=val)`
3. Ang halaga ng `options.default_shots`

Para sa anumang Estimator PUB:

1. Mga precision na may float na halaga na nakapaloob sa PUB
2. Ang halaga ng `run(...,precision=val)`
3. Ang halaga ng `options.default_shots`
4. Ang halaga ng `options.default_precision`

Halimbawa:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### Maximum na oras ng pagpapatupad
Ang maximum na oras ng pagpapatupad (`max_execution_time`) ay naglilimita kung gaano katagal maaaring tumakbo ang isang job. Kung ang isang job ay lalampas sa limitasyon ng oras na ito, sapilitang iki-cancel ito. Ang halagang ito ay nalalapat sa mga indibidwal na job, kahit sila ay pinapatakbo sa job, session, o batch mode.

Ang halaga ay itinakda sa mga segundo, batay sa quantum time (hindi wall clock time), na siyang dami ng oras na nakatuon ang QPU sa pagpoproseso ng iyong job. Binabalewala ito kapag gumagamit ng local testing mode dahil ang mode na iyon ay hindi gumagamit ng quantum time.